# 09 • Multimodal RAG: ค้นภาพด้วย Caption Embeddings

เป้าหมาย: ภาพ → LLM อธิบายภาพ → Text Embedding → บันทึก index → ค้นภาพต้นฉบับ → LLM ตอบจากภาพ

เป็นการค้นภาพผ่านคำบรรยาย ไม่ใช่การสร้าง image embedding โดยตรง หากคำบรรยายตกหล่นรายละเอียด การค้นหาอาจผิดพลาด

[เปิด notebook นี้ใน Colab](https://colab.research.google.com/drive/1Ndbr7FwCftYtmh-0iW6TyQ8z34eIkotx?usp=sharing)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai numpy pillow

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


## อัปโหลดภาพตัวอย่าง
เลือก PNG/JPEG 2–3 ภาพจาก materials/mock_images อัปโหลดผ่าน Files pane ทางซ้ายก่อนรันได้ โดยคงชื่อไฟล์ 01_, 02_, 03_ หรือใช้ปุ่มเลือกไฟล์ใน output

ภาพและ index เก็บใน runtime ดาวน์โหลด image_index.zip หากต้องการเก็บไว้ ภาพจะส่งเข้า OpenAI ตอนสร้างคำบรรยายและตอนตอบ ใช้เฉพาะข้อมูลสมมติในการอบรม


In [ ]:
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
import base64, json, hashlib, io
import numpy as np

folder = Path("image_index")
folder.mkdir(exist_ok=True)
# อัปโหลดภาพตัวอย่างผ่าน Files pane ก่อนรันได้
sample_paths = sorted(Path(".").glob("0[123]_*.png"))
if sample_paths:
    uploaded = {p.name: p.read_bytes() for p in sample_paths}
else:
    try:
        from google.colab import files
    except ImportError:
        image_files = input("Image paths (separate with ;): ").split(";")
        paths = [Path(name.strip().strip('"')) for name in image_files]
        uploaded = {p.name: p.read_bytes() for p in paths}
    else:
        uploaded = files.upload()
if not 2 <= len(uploaded) <= 3:
    raise ValueError("เลือกภาพ PNG/JPEG จำนวน 2–3 ภาพ")
image_paths = []
for i, (name, data) in enumerate(uploaded.items(), 1):
    if Path(name).suffix.lower() not in {".png", ".jpg", ".jpeg"}:
        raise ValueError("รองรับเฉพาะ PNG/JPEG")
    path = folder / f"image_{i}.png"
    with PILImage.open(io.BytesIO(data)) as image:
        image.convert("RGB").save(path)
    image_paths.append(path)
    print("ภาพ", i, "ไฟล์ต้นฉบับ:", name)
    preview = PILImage.open(path)
    preview.thumbnail((360, 500))
    display(preview)

def image_content(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode()
    return {"type": "input_image", "image_url": "data:image/png;base64," + encoded,
            "detail": "high"}

In [ ]:
EMBED_MODEL = "text-embedding-3-small"
caption_prompt = "อธิบายภาพเป็นภาษาไทยสำหรับใช้ค้นหา ระบุประเภทเอกสาร งานที่เกี่ยวข้อง ข้อความสำคัญ Lot สถานะ QC พื้นที่ และความสัมพันธ์ในผังเท่าที่เห็น หากอ่านไม่ชัดให้ระบุ ห้ามเดา"
records = []
for path in image_paths:
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": caption_prompt}, image_content(path)
        ]}], max_output_tokens=500,
    )
    caption = response.output_text.strip()
    if not caption:
        raise ValueError("ไม่พบคำบรรยาย กรุณาตรวจ response ก่อนทำ index")
    records.append({"file": path.name, "caption": caption,
                    "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    print(path.name, caption, "\n")

result = client.embeddings.create(model=EMBED_MODEL, input=[r["caption"] for r in records])
vectors = np.array([item.embedding for item in result.data])
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
(folder / "records.json").write_text(json.dumps(
    {"caption_model": MODEL, "embedding_model": EMBED_MODEL, "records": records},
    ensure_ascii=False, indent=2), encoding="utf-8")
np.save(folder / "vectors.npy", vectors)
import shutil
shutil.make_archive("image_index", "zip", folder)
print("บันทึกภาพ คำบรรยาย และ vectors แล้ว — อย่ารันเซลล์นี้ซ้ำเมื่อเปลี่ยนเฉพาะคำถาม เพราะมีค่า API")
# หากต้องการเก็บ index ไว้หลัง runtime ปิด ให้รัน files.download("image_index.zip")

In [ ]:
# โหลด index ที่บันทึกไว้ ไม่สร้างคำบรรยายหรือ embedding ของภาพซ้ำ
saved = json.loads((folder / "records.json").read_text(encoding="utf-8"))
records = saved["records"]
vectors = np.load(folder / "vectors.npy", allow_pickle=False)
for record in records:
    assert hashlib.sha256((folder / record["file"]).read_bytes()).hexdigest() == record["sha256"]

question = "ค้นภาพผังที่แสดงพื้นที่พักรอ QC และ Lot ที่อยู่ในพื้นที่นั้น"
query = client.embeddings.create(model=saved["embedding_model"], input=[question]).data[0].embedding
query = np.array(query)
scores = vectors @ (query / np.linalg.norm(query))
ranking = np.argsort(scores)[::-1]
for i in ranking:
    print(records[i]["file"], "คะแนน cosine:", round(float(scores[i]), 3))

best = records[int(ranking[0])]
selected_path = folder / best["file"]
print("ภาพที่ค้นพบ:", selected_path)
print("คำบรรยายที่ใช้ค้น:", best["caption"])
display(PILImage.open(selected_path))
print("ระบบเลือกภาพอันดับแรกเสมอ คะแนนไม่ใช่ความน่าจะเป็น ต้องตรวจว่าภาพเกี่ยวข้องจริง")

In [ ]:
answer_question = "จากภาพที่ค้นพบ พื้นที่พักรอ QC อยู่ระหว่างพื้นที่ใด และ Lot ใดอยู่ในพื้นที่นั้น?"
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": answer_question + " ตอบภาษาไทยจากภาพนี้เท่านั้น ระบุชื่อไฟล์ " + best["file"] + " หากภาพไม่เกี่ยวข้องหรืออ่านไม่ชัดให้บอก ห้ามเดา"},
        image_content(selected_path),
    ]}], max_output_tokens=500,
)
print(response.output_text)

## ลองทำด้วยตนเอง
1. เปลี่ยนคำถามค้นหาเป็นใบรับน้ำยางหรือการประสานงานขนส่ง แล้วรันเฉพาะเซลล์ค้นหาและตอบ
2. ตรวจคำบรรยายเทียบกับภาพต้นฉบับ มีรายละเอียดใดตกหล่นหรือผิด?
3. ถามเรื่องที่ไม่มีในภาพ เช่น ตารางวันหยุด ตรวจว่าระบบยังเลือกภาพอันดับแรกหรือไม่ และอธิบายว่าต้องมีเกณฑ์ปฏิเสธผลค้นหาอย่างไร
4. เปรียบเทียบคำตอบจากคำบรรยายกับคำตอบจากภาพ โดยเฉพาะตำแหน่งในผัง

ข้อจำกัด: เป็น caption-based retrieval; ไม่ใช่ joint image/text embedding และยังไม่มีเกณฑ์ relevance ที่ผ่านการประเมิน ค่า API: สร้างคำบรรยาย 1 request ต่อภาพ, embedding คำบรรยาย 1 batch, embedding คำถาม 1 request และตอบจากภาพ 1 request ต่อการค้นหา
